# DDPM Training for Bearing Health Indicator
Welcome! Use this notebook to train the DDPM model on Google Colab using a GPU.


### Step 1: Upload Data
Upload your IMS dataset to Colab. You can click the 'Files' icon on the left, create a folder named `data/raw`, and upload the files there. Alternatively, you can mount your Google Drive.


## 1. Dependencies and Dataset


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

class IMSDataset(Dataset):
    def __init__(self, data_dir, mode='train', segment_len=4096, channel=0):
        """
        Args:
            data_dir (str): Path to data directory (e.g., 'data/raw').
            mode (str): 'train' (first 500 files) or 'test' (all files).
            segment_len (int): Length of each segment (default: 4096).
            channel (int): Column index to extract (default: 0 for Bearing 3/4 based on user input, usually 0 is fine). 
                           Note: IMS dataset structure varies. Assuming 1st column is the target.
        """
        self.data_dir = Path(data_dir)
        self.mode = mode
        self.segment_len = segment_len
        self.channel = channel
        
        # 1. Loader: Sort files by timestamp
        self.files = sorted([f for f in self.data_dir.iterdir() if f.is_file()])
        
        # 2. Split: Train (first 500), Test (all/remaining)
        # "Train only on the 'healthy' data (first 20% of files)" - user prompt
        # User prompt also said: "train: Return segments from the first 500 files"
        if mode == 'train':
            self.files = self.files[:500]
        # test mode uses all files per instructions
        
        self.segments = []
        self._load_and_segment()

    def _load_and_segment(self):
        """
        Loads files, normalizes, and segments the data.
        """
        print(f"Loading {len(self.files)} files for {self.mode} mode...")
        
        for file_path in self.files:
            try:
                # IMS data is tab separated, no header
                df = pd.read_csv(file_path, sep='\t', header=None)
                signal = df.iloc[:, self.channel].values.astype(np.float32)
                
                # 4. Normalization: Normalize to [-1, 1]
                # Note: MinMax should ideally be fitted on training set global stats, 
                # but per-sample or per-file normalization is common in some GAN setups.
                # However, for a digital twin, global normalization is safer.
                # Here, following "Normalize signals to range [-1,1]" instruction literally per file for simplicity 
                # or we can do it on the segment. Let's do it per file to maintain relative amplitude within file.
                scaler = MinMaxScaler(feature_range=(-1, 1))
                signal = scaler.fit_transform(signal.reshape(-1, 1)).flatten()

                # 2. Segmentation
                # "Implementing a sliding window or random crop to split the 20,480-point signal into segments of length 4096."
                # We will use non-overlapping sliding window for simplicity and coverage.
                num_segments = len(signal) // self.segment_len
                for i in range(num_segments):
                    start = i * self.segment_len
                    end = start + self.segment_len
                    segment = signal[start:end]
                    self.segments.append(segment)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        # Return [1, 4096] tensor
        segment = self.segments[idx]
        return torch.tensor(segment, dtype=torch.float32).unsqueeze(0)



## 2. Diffusion Scheduler


In [ ]:
import torch
import torch.nn as nn


class DDPMScheduler(nn.Module):
    """
    DDPM Noise Scheduler with linear beta schedule.

    Forward process: x_t = sqrt(alpha_hat_t) * x0 + sqrt(1 - alpha_hat_t) * eps
    Reverse process: single-step DDPM posterior mean denoising.
    """

    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02):
        super().__init__()
        self.T = T

        # Linear beta schedule
        beta = torch.linspace(beta_start, beta_end, T)          # [T]
        alpha = 1.0 - beta                                        # [T]
        alpha_hat = torch.cumprod(alpha, dim=0)                  # [T]  ᾱ_t

        # Pre-compute commonly used quantities and register as buffers
        self.register_buffer("beta", beta)
        self.register_buffer("alpha", alpha)
        self.register_buffer("alpha_hat", alpha_hat)
        self.register_buffer("sqrt_alpha_hat", alpha_hat.sqrt())
        self.register_buffer("sqrt_one_minus_alpha_hat", (1.0 - alpha_hat).sqrt())
        self.register_buffer("sqrt_alpha", alpha.sqrt())
        self.register_buffer("sqrt_recip_alpha", alpha.rsqrt())

    def add_noise(self, x0: torch.Tensor, eps: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Forward diffusion: x_t = sqrt(ᾱ_t) * x0 + sqrt(1-ᾱ_t) * eps

        Args:
            x0:  Clean signal  [B, 1, L]
            eps: Gaussian noise [B, 1, L]
            t:   Timestep indices (long) [B]

        Returns:
            x_t: Noisy signal [B, 1, L]
        """
        sqrt_ah = self.sqrt_alpha_hat[t].view(-1, 1, 1)          # [B, 1, 1]
        sqrt_one_minus_ah = self.sqrt_one_minus_alpha_hat[t].view(-1, 1, 1)
        return sqrt_ah * x0 + sqrt_one_minus_ah * eps

    @torch.no_grad()
    def denoise_step(self, model, x_t: torch.Tensor, t_scalar: int) -> torch.Tensor:
        """
        One reverse denoising step (DDPM posterior mean).

        Args:
            model:    Trained UNet1D noise predictor
            x_t:      Noisy signal at timestep t  [B, 1, L]
            t_scalar: Current timestep (Python int, same for entire batch)

        Returns:
            x_{t-1}: Less noisy signal [B, 1, L]
        """
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t_scalar, device=x_t.device, dtype=torch.long)

        eps_hat = model(x_t, t_tensor)                          # predicted noise

        beta_t = self.beta[t_scalar]
        alpha_t = self.alpha[t_scalar]
        alpha_hat_t = self.alpha_hat[t_scalar]
        sqrt_recip_alpha_t = self.sqrt_recip_alpha[t_scalar]
        sqrt_one_minus_ah_t = self.sqrt_one_minus_alpha_hat[t_scalar]

        # DDPM posterior mean: μ_θ(x_t, t)
        mean = sqrt_recip_alpha_t * (x_t - (beta_t / sqrt_one_minus_ah_t) * eps_hat)

        if t_scalar == 0:
            return mean
        else:
            # Add posterior variance noise: σ_t = sqrt(β_t)
            noise = torch.randn_like(x_t)
            sigma = beta_t.sqrt()
            return mean + sigma * noise



## 3. DDPM Models (UNet1D)


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Sinusoidal Time Embedding
# ---------------------------------------------------------------------------

class SinusoidalEmbedding(nn.Module):
    """Maps scalar timestep t → d-dimensional sinusoidal embedding."""

    def __init__(self, dim: int = 256):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: [B] long timestep indices
        Returns:
            emb: [B, dim]
        """
        device = t.device
        half = self.dim // 2
        freq = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )                                                      # [half]
        args = t[:, None].float() * freq[None, :]             # [B, half]
        emb = torch.cat([args.sin(), args.cos()], dim=-1)     # [B, dim]
        return emb


# ---------------------------------------------------------------------------
# Residual Block
# ---------------------------------------------------------------------------

class ResBlock1D(nn.Module):
    """
    GroupNorm → SiLU → Conv1d → GroupNorm → SiLU → Conv1d + residual skip.
    Time embedding injected via a linear projection added after the first norm.
    """

    def __init__(self, in_ch: int, out_ch: int, time_dim: int = 256, groups: int = 8):
        super().__init__()
        self.time_proj = nn.Linear(time_dim, out_ch)

        self.norm1 = nn.GroupNorm(min(groups, in_ch), in_ch)
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)

        self.norm2 = nn.GroupNorm(min(groups, out_ch), out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)

        # Skip connection (1×1 conv if channels differ)
        self.skip = (
            nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x:     [B, in_ch, L]
            t_emb: [B, time_dim]
        Returns:
            out:   [B, out_ch, L]
        """
        h = F.silu(self.norm1(x))
        h = self.conv1(h)

        # Inject time embedding
        h = h + self.time_proj(F.silu(t_emb))[:, :, None]

        h = F.silu(self.norm2(h))
        h = self.conv2(h)

        return h + self.skip(x)


# ---------------------------------------------------------------------------
# UNet1D
# ---------------------------------------------------------------------------

class UNet1D(nn.Module):
    """
    1D U-Net noise predictor for DDPM on vibration signals.

    Architecture:
      Input  : [B, 1, 4096]
      Encoder stages (3):
        stage 0: ResBlock(64  → 128), Downsample(128) — L: 4096 → 2048
        stage 1: ResBlock(128 → 256), Downsample(256) — L: 2048 → 1024
        stage 2: ResBlock(256 → 512), Downsample(512) — L: 1024 → 512
      Bottleneck: ResBlock(512 → 512) at L=512
      Decoder stages (3):
        stage 0: Upsample(512→256), cat(skip=512) → 768, ResBlock(768→256) — L: 512 → 1024
        stage 1: Upsample(256→128), cat(skip=256) → 384, ResBlock(384→128) — L: 1024 → 2048
        stage 2: Upsample(128→64),  cat(skip=128) → 192, ResBlock(192→64)  — L: 2048 → 4096
      Output : GroupNorm → SiLU → Conv1d(64→1) → [B, 1, 4096]

    Skip connections carry the encoder ResBlock output BEFORE downsampling.
    """

    def __init__(self, in_channels: int = 1, time_dim: int = 256):
        super().__init__()

        # Channel progression: input_proj output, then encoder stages
        base_ch   = 64
        enc_chs   = [128, 256, 512]   # output channels per encoder stage
        self.enc_in_ch = [base_ch] + enc_chs[:-1]   # [64, 128, 256]

        # Time embedding MLP
        self.time_embed = nn.Sequential(
            SinusoidalEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

        # Input projection: 1 → 64
        self.input_proj = nn.Conv1d(in_channels, base_ch, kernel_size=3, padding=1)

        # Encoder blocks (each: in_ch → out_ch) + stride-2 downsamplers
        self.enc_blocks = nn.ModuleList([
            ResBlock1D(in_c, out_c, time_dim)
            for in_c, out_c in zip(self.enc_in_ch, enc_chs)
        ])
        self.downsamplers = nn.ModuleList([
            nn.Conv1d(out_c, out_c, kernel_size=4, stride=2, padding=1)
            for out_c in enc_chs
        ])

        # Bottleneck
        self.bottleneck = ResBlock1D(enc_chs[-1], enc_chs[-1], time_dim)

        # Decoder: upsample then concat with skip, then ResBlock
        # skip channels come from encoder outputs (reversed): [512, 256, 128]
        # upsampler input channels (reversed enc_chs):        [512, 256, 128]
        # upsampler output channels:                          [256, 128,  64]
        dec_out_chs = [256, 128, 64]
        skip_chs    = list(reversed(enc_chs))   # [512, 256, 128]
        up_in_chs   = list(reversed(enc_chs))   # [512, 256, 128]  (from prev dec stage out or bottleneck)
        # After up+concat: in_ch = dec_out_ch + skip_ch
        dec_in_chs  = [d + s for d, s in zip(dec_out_chs, skip_chs)]  # [768, 384, 192]

        self.upsamplers = nn.ModuleList([
            nn.ConvTranspose1d(up_c, out_c, kernel_size=4, stride=2, padding=1)
            for up_c, out_c in zip(up_in_chs, dec_out_chs)
        ])
        self.dec_blocks = nn.ModuleList([
            ResBlock1D(in_c, out_c, time_dim)
            for in_c, out_c in zip(dec_in_chs, dec_out_chs)
        ])

        # Output projection: 64 → 1
        self.output_proj = nn.Sequential(
            nn.GroupNorm(8, dec_out_chs[-1]),
            nn.SiLU(),
            nn.Conv1d(dec_out_chs[-1], in_channels, kernel_size=1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Noisy signal  [B, 1, L]   (L = 4096)
            t: Timestep      [B] (long)
        Returns:
            Predicted noise  [B, 1, L]
        """
        t_emb = self.time_embed(t)            # [B, time_dim]

        # Input projection
        h = self.input_proj(x)                # [B, 64, L]

        # Encoder — save skip connections BEFORE downsampling
        skips = []
        for enc, down in zip(self.enc_blocks, self.downsamplers):
            h = enc(h, t_emb)                 # [B, out_ch, L]
            skips.append(h)                   # save before downsampling
            h = down(h)                       # [B, out_ch, L/2]

        # Bottleneck  [B, 512, L//8]
        h = self.bottleneck(h, t_emb)

        # Decoder — upsample, concat skip, ResBlock
        for up, dec, skip in zip(self.upsamplers, self.dec_blocks, reversed(skips)):
            h = up(h)
            # Handle potential size mismatch (odd-length signals)
            if h.shape[-1] != skip.shape[-1]:
                h = F.interpolate(h, size=skip.shape[-1])
            h = torch.cat([h, skip], dim=1)   # concat on channel axis
            h = dec(h, t_emb)

        return self.output_proj(h)            # [B, 1, L]



## 4. Training Logic


In [ ]:
"""
DDPM Training Loop for the IMS Bearing Dataset.

Usage (CLI):
    python -m src.train --n_epochs 100 --batch_size 32 --lr 2e-4 \
                        --data_dir data/raw --channel 0 --timesteps 1000

Usage (programmatic, called from app.py):
    from src.train import train
    train(args)  # args has attributes: n_epochs, batch_size, lr, data_dir, channel, timesteps
"""

import argparse
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader



def train(args) -> None:
    """
    Train a UNet1D DDPM on healthy bearing data (first 500 files).

    Args:
        args: Namespace or object with attributes:
              n_epochs, batch_size, lr, data_dir, channel, timesteps
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Train] Using device: {device}")

    # ------------------------------------------------------------------ #
    # 1. Dataset & DataLoader
    # ------------------------------------------------------------------ #
    dataset = IMSDataset(
        data_dir=args.data_dir,
        mode="train",
        channel=args.channel,
    )
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    print(f"[Train] Dataset: {len(dataset)} segments | Batches/epoch: {len(loader)}")

    # ------------------------------------------------------------------ #
    # 2. Model & Scheduler
    # ------------------------------------------------------------------ #
    timesteps = getattr(args, "timesteps", 1000)
    model = UNet1D(in_channels=1).to(device)
    scheduler = DDPMScheduler(T=timesteps).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    loss_fn = nn.MSELoss()

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Train] UNet1D parameters: {param_count:,}")

    # ------------------------------------------------------------------ #
    # 3. Training Loop
    # ------------------------------------------------------------------ #
    model.train()
    for epoch in range(1, args.n_epochs + 1):
        epoch_loss = 0.0
        for batch in loader:
            x0 = batch.to(device)              # [B, 1, 4096]

            # Sample random timesteps uniformly from {1, ..., T}
            t = torch.randint(1, timesteps, (x0.shape[0],), device=device, dtype=torch.long)

            # Sample Gaussian noise
            eps = torch.randn_like(x0)

            # Forward diffusion: x_t = sqrt(ᾱ_t) * x0 + sqrt(1-ᾱ_t) * eps
            x_t = scheduler.add_noise(x0, eps, t)

            # Predict noise
            eps_hat = model(x_t, t)

            # MSE loss between true and predicted noise
            loss = loss_fn(eps_hat, eps)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        print(f"[Train] Epoch [{epoch:>4}/{args.n_epochs}]  Loss: {avg_loss:.6f}")

    # ------------------------------------------------------------------ #
    # 4. Save checkpoint
    # ------------------------------------------------------------------ #
    save_path = "unet_diffusion.pth"
    torch.save(model.state_dict(), save_path)
    print(f"[Train] Model saved → {save_path}")


# ------------------------------------------------------------------ #
# CLI entry point
# ------------------------------------------------------------------ #
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train DDPM on IMS bearing data")
    parser.add_argument("--n_epochs",   type=int,   default=100,    help="Number of training epochs")
    parser.add_argument("--batch_size", type=int,   default=32,     help="Batch size")
    parser.add_argument("--lr",         type=float, default=2e-4,   help="Adam learning rate")
    parser.add_argument("--data_dir",   type=str,   default="data/raw", help="Path to raw IMS data")
    parser.add_argument("--channel",    type=int,   default=0,      help="Bearing channel index (0-indexed)")
    parser.add_argument("--timesteps",  type=int,   default=1000,   help="Number of diffusion timesteps T")
    args = parser.parse_args()
    train(args)



## 5. Start Training
Make sure your runtime type is set to **T4 GPU** (Runtime -> Change runtime type). Then run the cell below to start training.


In [ ]:
class ColabArgs:
    n_epochs = 100
    batch_size = 32
    lr = 2e-4
    data_dir = "data/raw"  # <--- Update this to your data path if different
    channel = 0
    timesteps = 1000

import os
if not os.path.exists(ColabArgs.data_dir) or not os.listdir(ColabArgs.data_dir):
    print(f"Data directory not found or empty at: {ColabArgs.data_dir}!")
    print("Please upload your data files there first.")
else:
    print("Data found. Starting training...")
    train(ColabArgs())
    print("\nTraining complete! You can now download 'unet_diffusion.pth' from the Files pane on the left.")

